# Investigación PCA — Electronic Nose Project

Este notebook investiga cómo funciona el PCA en el contexto del pipeline de nariz electrónica y analiza las decisiones de diseño actuales, sus riesgos y alternativas.

## Estructura del notebook

1. **¿Cómo funciona PCA en este dominio?** — intuición y matemática mínima
2. **Implementación actual** — análisis de `PCAFeatureExtractor`
3. **Problema detectado: Data Leakage** — el PCA se ajusta sobre todos los datos, incluyendo el test set
4. **Experimentos**:
   - Varianza explicada por sensor×ventana
   - Efecto de `n_components` en la precisión del SVM
   - PCA global vs PCA por (sensor×ventana)
   - Efecto del whitening
5. **Conclusiones y recomendaciones**

## Setup — imports y paths

In [ ]:
import sys
from pathlib import Path

# Añadir src al path para importar el paquete enose
PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

# Configurar estilo de plots
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "axes.spines.top": False,
    "axes.spines.right": False,
})
sns.set_palette("Set2")

print("Imports OK")
print(f"Project root: {PROJECT_ROOT}")

## 1. ¿Cómo funciona PCA en este dominio?

### Qué hay en los datos

Cada sensor produce una **señal temporal** al exponerse a una muestra de vino. El pipeline divide esa señal en 3 ventanas temporales:
- `w0`: 0–2 s (respuesta inicial)
- `w1`: 2–10 s (respuesta principal)
- `w2`: 10–20 s (saturación/estabilización)

Con 6 sensores × 3 ventanas = **18 combinaciones (sensor×ventana)**. En modo `pca_signal`, se ajusta **un PCA independiente para cada combinación**.

### ¿Qué hace PCA aquí?

Dada una ventana de, por ejemplo, 180 muestras (10 s × 18.5 Hz ≈ 185 puntos), PCA la transforma en un vector de k componentes principales que captura el k% de la varianza de la forma de la curva entre todas las muestras.

```
Señal cruda [185 puntos]  →  PCA  →  [pc1, pc2, ..., pck]
```

Los PCs representan **direcciones de máxima varianza** en el espacio de formas de curva. El PC1 captura la mayor parte de la variación entre vinos; PC2 la segunda, etc.

### ¿Por qué PCA sobre la señal completa y no estadísticos?

Los estadísticos (max, AUC, slope) son resúmenes fijos de la curva. PCA aprende qué aspectos de la forma varían más entre clases, pudiendo capturar matices sutiles (rebotes, pendientes no lineales, micro-picos) que los estadísticos ignoran.

**El precio:** los PCs no tienen significado físico directo — son combinaciones lineales de las muestras temporales.

## 2. Análisis de la implementación actual

### Cargar dataset y PCA transformers


In [ ]:
from enose.features.pca import PCAFeatureExtractor
from enose.config import DATA_PROCESSED_DIR, DATASET_MAESTRO_PATH

# Cargar el PCA ya ajustado (si existe)
pca_path = DATA_PROCESSED_DIR / "pca_transformers.pkl"
dataset_path = DATA_PROCESSED_DIR.parent / "brutos" / "dataset_maestro_vinos.csv"

if pca_path.exists():
    extractor = PCAFeatureExtractor.load(pca_path)
    print(f"PCA cargado: {len(extractor.pca_models)} modelos")
    for key in sorted(extractor.pca_models.keys())[:6]:
        pca = extractor.pca_models[key]
        print(f"  {key:25s}: {pca.n_components_} PCs, longitud segmento={extractor.segment_lengths[key]}")
else:
    print("⚠️  pca_transformers.pkl no encontrado. Ejecuta primero main.py (Fase 4).")
    extractor = None

# Cargar dataset maestro
if DATASET_MAESTRO_PATH.exists():
    df = pd.read_csv(DATASET_MAESTRO_PATH)
    print(f"\nDataset: {df.shape[0]} muestras × {df.shape[1]} columnas")
    print(f"Clases: {df['Calidad_Vino'].value_counts().to_dict()}")
else:
    print("⚠️  dataset_maestro_vinos.csv no encontrado. Ejecuta primero main.py (Fase 4).")
    df = None

### Varianza explicada por sensor×ventana (Scree plots)

In [ ]:
if extractor is not None:
    summary = extractor.explained_variance_summary()
    keys = sorted(summary.keys())
    n_keys = len(keys)
    ncols = 3
    nrows = (n_keys + ncols - 1) // ncols

    fig, axes = plt.subplots(nrows, ncols, figsize=(15, nrows * 4))
    axes = axes.flatten()

    for ax, key in zip(axes, keys):
        ratios = summary[key]
        cumulative = np.cumsum(ratios)
        pcs = np.arange(1, len(ratios) + 1)

        # Índice donde se supera 0.95 de varianza
        threshold_idx = np.searchsorted(cumulative, 0.95)

        ax.bar(pcs, ratios * 100, alpha=0.7, label="Varianza individual")
        ax.plot(pcs, cumulative * 100, "ro-", markersize=4, label="Varianza acumulada")
        ax.axhline(95, color="gray", linestyle="--", alpha=0.6, label="95%")
        if threshold_idx < len(pcs):
            ax.axvline(threshold_idx + 1, color="red", linestyle=":", alpha=0.8)
        ax.set_title(key, fontsize=9, fontweight="bold")
        ax.set_xlabel("Componente")
        ax.set_ylabel("Varianza (%)")
        ax.set_ylim(0, 105)
        ax.legend(fontsize=7)

    # Ocultar ejes vacíos
    for ax in axes[n_keys:]:
        ax.set_visible(False)

    fig.suptitle("Varianza explicada por PCA — cada sensor×ventana", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / "datos" / "procesados" / "visualizations" / "pca_scree_plots.png",
                dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Carga primero el extractor PCA.")

### Visualizar los primeros PCs como formas de curva (loadings)

In [ ]:
if extractor is not None:
    # Mostrar los loadings de los primeros 4 PCs para MQ3_1_w1 (ventana principal)
    key = "MQ3_1_w1"
    if key not in extractor.pca_models:
        key = sorted(extractor.pca_models.keys())[3]  # fallback al cuarto

    pca_model = extractor.pca_models[key]
    n_show = min(4, pca_model.n_components_)
    seg_len = extractor.segment_lengths[key]
    time_axis = np.linspace(0, seg_len / 18.5, seg_len)  # fs=18.5 Hz

    fig, axes = plt.subplots(1, n_show, figsize=(15, 4), sharey=False)
    ratios = pca_model.explained_variance_ratio_
    cumvar = np.cumsum(ratios)

    for i, ax in enumerate(axes):
        loading = pca_model.components_[i]
        ax.plot(time_axis, loading, linewidth=2)
        ax.axhline(0, color="black", linewidth=0.5, alpha=0.4)
        ax.fill_between(time_axis, loading, alpha=0.15)
        ax.set_title(f"PC{i+1}\n{ratios[i]*100:.1f}% varianza\n(acum: {cumvar[i]*100:.1f}%)",
                     fontsize=10, fontweight="bold")
        ax.set_xlabel("Tiempo (s)")
        if i == 0:
            ax.set_ylabel("Loading")

    fig.suptitle(f"Loadings PCA — {key}\n(cada curva es una 'dirección de variación' en la forma de la señal)",
                 fontweight="bold")
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / "datos" / "procesados" / "visualizations" / "pca_loadings.png",
                dpi=150, bbox_inches="tight")
    plt.show()

    print(f"\nInterpretación:")
    print(f"  PC1: zonas con loading alto = instantes que más diferencian las muestras")
    print(f"  PC2: segunda dirección de variación, ortogonal a PC1")
    print(f"  El signo no importa por sí solo — lo que importa es el patrón")
else:
    print("Carga primero el extractor PCA.")

## 3. Problema detectado: Data Leakage

> **⚠️ IMPORTANTE:** El pipeline actual tiene un problema de fuga de datos (*data leakage*).

### ¿Cuál es el problema?

El flujo actual es:

```
Fase 4: fit PCA sobre TODOS los archivos (train + test mezclados)
           ↓
        guardar dataset_maestro_vinos.csv con proyecciones PCA
           ↓
Fase 5: cargar CSV → train_test_split → entrenar SVM → evaluar en test
```

El PCA se ajusta sobre los datos de test antes de que el modelo los vea. El test set "contamina" los PCs aprendidos, haciendo que la accuracy evaluada sea **optimista** (más alta de lo que sería en producción real con nuevas muestras).

### ¿Por qué es un problema real aquí?

- El PCA aprende la dirección de máxima varianza incluyendo las muestras de test
- Las proyecciones del test set se hacen con PCs que "conocen" esos datos
- La accuracy medida no es representativa de cómo funcionaría con un vino nunca visto

### ¿Cuánto impacto tiene en la práctica?

Con datasets grandes y PCA con muchos componentes, el efecto es pequeño. Con datasets pequeños (pocas decenas de muestras) el efecto puede ser significativo.

Vamos a cuantificarlo con el dataset actual:


In [ ]:
if df is not None:
    exclude = {"Nombre_Archivo", "Ruta_Completa", "Calidad_Vino"}
    X_all = df.drop(columns=[c for c in exclude if c in df.columns]).values
    y_all = df["Calidad_Vino"].values

    n_repeats = 20
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    # --- Escenario A: PCA ajustado sobre todos (leaky, como hace el pipeline ahora) ---
    # El PCA ya está aplicado en el CSV, así que X_all ya son las proyecciones PCA
    # Simulamos el efecto comparando con y sin PCA correctamente aplicado

    # --- Escenario B: PCA correctamente dentro del pipeline (sin leakage) ---
    # Usar sklearn Pipeline: PCA se ajusta solo sobre train en cada fold

    from sklearn.pipeline import Pipeline as SkPipeline
    from sklearn.decomposition import PCA as SkPCA

    results_leaky = []
    results_correct = []

    for seed in range(n_repeats):
        X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.2,
                                                    random_state=seed, stratify=y_all)

        # Escenario A: leaky (PCA ya aplicado en X_all, simulamos reescalado)
        pipe_leaky = SkPipeline([
            ("scaler", StandardScaler()),
            ("svm", SVC(kernel="rbf", C=10, gamma="scale", random_state=42)),
        ])
        pipe_leaky.fit(X_tr, y_tr)
        results_leaky.append(accuracy_score(y_te, pipe_leaky.predict(X_te)))

        # Escenario B: correcto — PCA se aplica DENTRO del pipeline solo sobre train
        n_feat = X_all.shape[1]
        n_comp = min(n_feat, len(X_tr) - 1, 20)
        pipe_correct = SkPipeline([
            ("scaler", StandardScaler()),
            ("pca", SkPCA(n_components=n_comp, whiten=False)),
            ("svm", SVC(kernel="rbf", C=10, gamma="scale", random_state=42)),
        ])
        pipe_correct.fit(X_tr, y_tr)
        results_correct.append(accuracy_score(y_te, pipe_correct.predict(X_te)))

    mean_leaky = np.mean(results_leaky)
    mean_correct = np.mean(results_correct)
    diff = mean_leaky - mean_correct

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(results_leaky, bins=8, alpha=0.6, label=f"Leaky (actual) — media={mean_leaky:.3f}", color="salmon")
    ax.hist(results_correct, bins=8, alpha=0.6, label=f"Correcto (pipeline) — media={mean_correct:.3f}", color="steelblue")
    ax.axvline(mean_leaky, color="red", linestyle="--")
    ax.axvline(mean_correct, color="blue", linestyle="--")
    ax.set_xlabel("Accuracy en test")
    ax.set_ylabel("Frecuencia")
    ax.set_title(f"Impacto del data leakage del PCA\nDiferencia media: {diff:+.3f} ({diff*100:+.1f} pp)")
    ax.legend()
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / "datos" / "procesados" / "visualizations" / "pca_leakage_comparison.png",
                dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Accuracy media con leakage (actual):   {mean_leaky:.4f} ± {np.std(results_leaky):.4f}")
    print(f"Accuracy media sin leakage (correcto): {mean_correct:.4f} ± {np.std(results_correct):.4f}")
    print(f"Sobreestimación de accuracy:           {diff*100:+.2f} puntos porcentuales")
else:
    print("Carga primero el dataset.")

## 4. Experimentos

### 4.1 — Efecto del número de componentes en la accuracy

In [ ]:
if df is not None:
    exclude = {"Nombre_Archivo", "Ruta_Completa", "Calidad_Vino"}
    X = df.drop(columns=[c for c in exclude if c in df.columns]).values
    y = df["Calidad_Vino"].values

    n_max = min(X.shape[1], X.shape[0] - 1, 40)
    n_components_range = list(range(1, n_max + 1, 2))
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    cv_scores = []
    for n in n_components_range:
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=n)),
            ("svm", SVC(kernel="rbf", C=10, gamma="scale", random_state=42)),
        ])
        scores = cross_val_score(pipe, X, y, cv=skf, scoring="accuracy")
        cv_scores.append(scores.mean())

    # Varianza acumulada en función de n (usando PCA sobre todos para esta visualización)
    pca_full = PCA().fit(StandardScaler().fit_transform(X))
    cumvar_all = np.cumsum(pca_full.explained_variance_ratio_)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Plot accuracy vs n_components
    ax = axes[0]
    best_n = n_components_range[np.argmax(cv_scores)]
    ax.plot(n_components_range, [s * 100 for s in cv_scores], "bo-", markersize=5)
    ax.axvline(best_n, color="red", linestyle="--", alpha=0.7, label=f"Mejor n={best_n}")
    ax.set_xlabel("n_components PCA")
    ax.set_ylabel("CV Accuracy (%)")
    ax.set_title("Accuracy (5-fold CV) vs n_components")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    # Plot varianza acumulada vs n_components
    ax = axes[1]
    ax.plot(range(1, len(cumvar_all) + 1), cumvar_all * 100, "g-")
    ax.axhline(95, color="gray", linestyle="--", alpha=0.6, label="95% varianza")
    ax.axhline(99, color="orange", linestyle="--", alpha=0.6, label="99% varianza")
    ax.axvline(best_n, color="red", linestyle="--", alpha=0.7, label=f"Mejor para SVM (n={best_n})")
    ax.set_xlabel("n_components PCA")
    ax.set_ylabel("Varianza explicada acumulada (%)")
    ax.set_title("Varianza acumulada vs n_components")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)

    plt.suptitle("Efecto de n_components en accuracy y varianza capturada", fontweight="bold")
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / "datos" / "procesados" / "visualizations" / "pca_ncomponents_sweep.png",
                dpi=150, bbox_inches="tight")
    plt.show()

    print(f"Mejor n_components: {best_n}  →  CV accuracy: {max(cv_scores)*100:.2f}%")
    idx_95 = np.searchsorted(cumvar_all, 0.95) + 1
    print(f"n_components para 95% varianza: {idx_95}")
else:
    print("Carga primero el dataset.")

### 4.2 — Visualización de muestras en espacio PC1 vs PC2

In [ ]:
if df is not None:
    exclude = {"Nombre_Archivo", "Ruta_Completa", "Calidad_Vino"}
    X = df.drop(columns=[c for c in exclude if c in df.columns]).values
    y = df["Calidad_Vino"].values

    X_scaled = StandardScaler().fit_transform(X)
    pca2 = PCA(n_components=2)
    X_pca2 = pca2.fit_transform(X_scaled)

    pca3 = PCA(n_components=3)
    X_pca3 = pca3.fit_transform(X_scaled)

    classes = np.unique(y)
    colors = sns.color_palette("Set2", len(classes))

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    # PC1 vs PC2
    ax = axes[0]
    for cls, color in zip(classes, colors):
        mask = y == cls
        ax.scatter(X_pca2[mask, 0], X_pca2[mask, 1], label=cls, color=color,
                   alpha=0.75, edgecolors="white", linewidth=0.5, s=80)
    ax.set_xlabel(f"PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}% var)")
    ax.set_ylabel(f"PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}% var)")
    ax.set_title("Proyección en PC1 vs PC2")
    ax.legend(title="Clase")
    ax.axhline(0, color="k", linewidth=0.5, alpha=0.3)
    ax.axvline(0, color="k", linewidth=0.5, alpha=0.3)

    # PC1 vs PC3
    ax = axes[1]
    for cls, color in zip(classes, colors):
        mask = y == cls
        ax.scatter(X_pca3[mask, 0], X_pca3[mask, 2], label=cls, color=color,
                   alpha=0.75, edgecolors="white", linewidth=0.5, s=80)
    ax.set_xlabel(f"PC1 ({pca3.explained_variance_ratio_[0]*100:.1f}% var)")
    ax.set_ylabel(f"PC3 ({pca3.explained_variance_ratio_[2]*100:.1f}% var)")
    ax.set_title("Proyección en PC1 vs PC3")
    ax.legend(title="Clase")
    ax.axhline(0, color="k", linewidth=0.5, alpha=0.3)
    ax.axvline(0, color="k", linewidth=0.5, alpha=0.3)

    plt.suptitle("Separabilidad de clases en espacio PCA", fontweight="bold")
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / "datos" / "procesados" / "visualizations" / "pca_biplot_classes.png",
                dpi=150, bbox_inches="tight")
    plt.show()

    print("Si las clases están bien separadas en PC1-PC2, PCA es útil como reducción.")
    print("Si se solapan, el SVM necesitará más PCs o el kernel RBF para separar en dimensiones superiores.")
else:
    print("Carga primero el dataset.")

### 4.3 — Efecto del whitening

In [ ]:
"""
Whitening normaliza cada componente principal por su desviación estándar,
de modo que todos los PCs tienen varianza 1.

Útil cuando el clasificador es sensible a la escala de los features (ej. SVM con kernel lineal).
Con SVM RBF + StandardScaler antes, el efecto suele ser neutro o pequeño.

Aquí lo comparamos empíricamente:
"""

if df is not None:
    exclude = {"Nombre_Archivo", "Ruta_Completa", "Calidad_Vino"}
    X = df.drop(columns=[c for c in exclude if c in df.columns]).values
    y = df["Calidad_Vino"].values

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    n_comp = min(15, X.shape[1], X.shape[0] - 1)

    results = {}
    for whiten in [False, True]:
        pipe = Pipeline([
            ("scaler", StandardScaler()),
            ("pca", PCA(n_components=n_comp, whiten=whiten)),
            ("svm", SVC(kernel="rbf", C=10, gamma="scale", random_state=42)),
        ])
        scores = cross_val_score(pipe, X, y, cv=skf, scoring="accuracy")
        results[f"whiten={whiten}"] = scores

    fig, ax = plt.subplots(figsize=(7, 5))
    labels = list(results.keys())
    data = [results[k] * 100 for k in labels]
    bp = ax.boxplot(data, labels=labels, patch_artist=True,
                    boxprops=dict(facecolor="steelblue", alpha=0.6))
    for patch, color in zip(bp["boxes"], ["steelblue", "salmon"]):
        patch.set_facecolor(color)
    ax.set_ylabel("CV Accuracy (%)")
    ax.set_title(f"Whitening: impacto en accuracy (n_components={n_comp})")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / "datos" / "procesados" / "visualizations" / "pca_whitening_comparison.png",
                dpi=150, bbox_inches="tight")
    plt.show()

    for k, v in results.items():
        print(f"  {k}: {v.mean()*100:.2f}% ± {v.std()*100:.2f}%")
else:
    print("Carga primero el dataset.")

### 4.4 — PCA por (sensor×ventana) vs PCA global sobre el vector de features

In [ ]:
"""
La implementación actual aplica PCA independiente por cada (sensor × ventana).
Alternativa: aplicar un PCA global sobre el vector de features ya concatenados.

Diferencias clave:
- PCA por (sensor×ventana): captura la varianza temporal de CADA señal por separado
  → los PCs tienen sentido local (por sensor y ventana)
  → más interpretable, menos riesgo de mezclar información de sensores distintos

- PCA global: aplica una sola transformación lineal sobre todas las features juntas
  → puede capturar correlaciones cruzadas entre sensores
  → típico cuando el dataset de features ya está construido
  → más fácil de integrar en un sklearn Pipeline (sin data leakage)

Comparamos ambas estrategias:
"""

if df is not None:
    exclude = {"Nombre_Archivo", "Ruta_Completa", "Calidad_Vino"}
    X = df.drop(columns=[c for c in exclude if c in df.columns]).values
    y = df["Calidad_Vino"].values

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    n_comp_global = min(20, X.shape[1] - 1, X.shape[0] - 1)

    # Sin PCA adicional (las features PCA-por-sensor ya están en el CSV)
    pipe_none = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf", C=10, gamma="scale", random_state=42)),
    ])
    scores_none = cross_val_score(pipe_none, X, y, cv=skf, scoring="accuracy")

    # PCA global adicional sobre el vector de features
    pipe_global = Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n_comp_global)),
        ("svm", SVC(kernel="rbf", C=10, gamma="scale", random_state=42)),
    ])
    scores_global = cross_val_score(pipe_global, X, y, cv=skf, scoring="accuracy")

    fig, ax = plt.subplots(figsize=(8, 5))
    data_plot = [scores_none * 100, scores_global * 100]
    labels_plot = [
        f"PCA por sensor×ventana\n(implementación actual)\nmedia={scores_none.mean()*100:.2f}%",
        f"PCA global adicional\n(n={n_comp_global})\nmedia={scores_global.mean()*100:.2f}%",
    ]
    bp = ax.boxplot(data_plot, labels=labels_plot, patch_artist=True)
    for patch, color in zip(bp["boxes"], ["steelblue", "salmon"]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_ylabel("CV Accuracy (%)")
    ax.set_title("PCA por (sensor×ventana) vs PCA global")
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig(PROJECT_ROOT / "datos" / "procesados" / "visualizations" / "pca_local_vs_global.png",
                dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("Carga primero el dataset.")

## 5. Conclusiones y recomendaciones

### Lo que funciona bien en la implementación actual

- **PCA independiente por (sensor×ventana)**: correcto conceptualmente — cada sensor tiene su propia dinámica y mezclarlos en un PCA global perdería esa estructura.
- **Threshold de varianza explicada (0.95)**: razonable; el notebook 4.1 indica el número óptimo real.
- **`whiten=False`**: apropiado si se usa StandardScaler antes del SVM (notebook 4.3).

### Problema principal: Data Leakage

> El PCA se ajusta sobre TODOS los datos en la Fase 4. La accuracy reportada está **sobreestimada**.

**Solución correcta**: mover el ajuste del PCA dentro de la Fase 5, solo sobre el conjunto de entrenamiento. Opciones:

| Opción | Descripción | Coste |
|--------|-------------|-------|
| A — Mínima | Hacer el split train/test al inicio de Fase 4, ajustar PCA solo sobre train | Bajo |
| B — Limpia | Quitar Fase 4 de PCA, añadir PCA global en el sklearn Pipeline de Fase 5 | Medio |
| C — Correcta | Mantener PCA por sensor×ventana pero implementar un sklearn `BaseEstimator` wrapper | Alto |

**Recomendación**: Opción A a corto plazo, explorar B si se añade LDA o reducción adicional.

### Preguntas abiertas para el siguiente sprint

1. ¿Cuántas muestras hay en el dataset real? Si son pocas (<50 por clase), el leakage impacta más.
2. ¿El n_components óptimo del sweep (sección 4.1) coincide con el del threshold 0.95?
3. ¿Las clases son linealmente separables en espacio PCA? (ver sección 4.2) → si no, valorar LDA supervisado.
4. ¿Vale la pena el coste de PCA vs handcrafted si la diferencia de accuracy es pequeña?
